In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/feature_engineered_dataset.csv")
print("Loaded shape:", df.shape)

Loaded shape: (10324, 38)


In [16]:
id_columns = ["ID", "Project Code", "PQ #", "PO / SO #", "ASN/DN #"]
df = df.drop(columns=id_columns)
print("Shape:", df.shape)

Shape: (10324, 33)


In [17]:
leakage_columns = [
    "Delivered to Client Date",
    "Delivery Recorded Date",
    "Delay Days",  # we keep "Is Delayed" as the target instead
]

df = df.drop(columns=leakage_columns)

print("Shape:", df.shape)

Shape: (10324, 30)


In [18]:
raw_date_columns = [
    "Scheduled Delivery Date",
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date",
]

df = df.drop(columns=raw_date_columns)

print("Shape:", df.shape)

Shape: (10324, 27)


In [19]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

# "Is Delayed" is our target, not a feature -- keep it separate from the numeric feature list
if "Is Delayed" in numeric_columns:
    numeric_columns.remove("Is Delayed")

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))
print(categorical_columns)

Numeric columns: 11
Categorical columns: 15
['Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Manufacturing Site', 'First Line Designation']


C:\Users\tanvi\AppData\Local\Temp\ipykernel_19420\79160878.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns.tolist()


In [20]:
HIGH_CARDINALITY_THRESHOLD = 25  # if a column has more unique values than this, we simplify it
TOP_N_CATEGORIES = 15            # how many top categories to keep per high-cardinality column

high_card_columns = [col for col in categorical_columns if df[col].nunique() > HIGH_CARDINALITY_THRESHOLD]
low_card_columns = [col for col in categorical_columns if col not in high_card_columns]

print("High-cardinality columns (will be simplified):", high_card_columns)
print()
print("Low-cardinality columns (safe to one-hot encode directly):", low_card_columns)

High-cardinality columns (will be simplified): ['Country', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Manufacturing Site']

Low-cardinality columns (safe to one-hot encode directly): ['Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'Product Group', 'Sub Classification', 'Dosage Form', 'First Line Designation']


In [21]:
for col in high_card_columns:
    top_categories = df[col].value_counts().nlargest(TOP_N_CATEGORIES).index
    df[col] = df[col].where(df[col].isin(top_categories), other="Other")

print("High-cardinality columns simplified!")
for col in high_card_columns:
    print(col, "-> now has", df[col].nunique(), "unique values")

High-cardinality columns simplified!
Country -> now has 16 unique values
Vendor -> now has 16 unique values
Item Description -> now has 16 unique values
Molecule/Test Type -> now has 16 unique values
Brand -> now has 16 unique values
Dosage -> now has 16 unique values
Manufacturing Site -> now has 16 unique values


In [22]:
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)

Shape before encoding: (10324, 27)
Shape after encoding: (10324, 158)


In [23]:
print("Any missing values left?")
print(df_encoded.isnull().sum().sum())

print()
print("Data types:")
print(df_encoded.dtypes.value_counts())

Any missing values left?
0

Data types:
bool       146
int64        6
float64      6
Name: count, dtype: int64


In [24]:
df_encoded.to_csv("../Dataset/preprocessed_dataset.csv", index=False)

print("Preprocessed dataset saved successfully!")
print("Final shape:", df_encoded.shape)

Preprocessed dataset saved successfully!
Final shape: (10324, 158)
